# Build the "Same Place?" game dataset

Exports `game_pairs.json` for the address_matcher React app (`~/Downloads/address_matcher`).

Shape (decided 2026-11-15):
- `models`: `[{id, name, threshold}]`; threshold is on the probability, 0.5 for all.
- `pairs`: `[{pair_id, address_a, address_b, label, neg_type, probs: {model_id: p}}]`; the keys of `probs` are the foreign key into `models[].id`.
- No stored pass/fail: the app derives it as `(p >= threshold) == label`.
- Test split only (models never trained on it), `overture_case2s` dataset.

Boilerplate (setup, loading pairs, loading every model's test probabilities) is filled in; the TODO cells are David's.

In [13]:
# Setup: imports and paths
import os
import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import json
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

from helpers.pairs import load_pairs, add_number_street
from helpers.features import get_processed_dfs

DATASET = "overture_case2s"
SEEDS = [0, 1, 2]
FEATURE_CACHE = "../model_baseline/case2s_training"   # built by model_baseline/07_backtest_case2s.ipynb
OUT_PATH = "game_pairs.json"

## 1. Pairs
All three splits are loaded because XGBoost has to be refitted on train; only `test` goes into the game.

In [14]:
# Load: case2s train / val / test pairs (same order the feature caches and saved probabilities were built from)
train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
test, test_label = load_pairs(DATASET, "test")
train, val, test = [add_number_street(df) for df in [train, val, test]]

test = test.assign(label = test_label.values)
test.groupby("neg_type", observed = True).size()

neg_type
case0      4386
case1     14448
case2        15
case2s     1053
case3     23942
dtype: int64

## 2. Model probabilities on test
One entry per model in `probs`: `{model_id: [p_seed0, p_seed1, p_seed2]}` (XGBoost is deterministic, so it gets one array).

In [15]:
# XGBoost (Step 1b): refit on the cached similarity features, same settings as run_against_models
train_X, val_X, test_X = get_processed_dfs([
    [train, f"{FEATURE_CACHE}/case2s_evaluation_train.parquet"],
    [val, f"{FEATURE_CACHE}/case2s_evaluation_val.parquet"],
    [test, f"{FEATURE_CACHE}/case2s_evaluation_test.parquet"],
])
xgb = XGBClassifier(random_state = 0).fit(train_X, train_label)

probs = {"xgb": [xgb.predict_proba(test_X)[:, 1]]}
probs

{'xgb': [array([0.8736193 , 0.99993587, 0.99993587, ..., 0.65009975, 0.00350178,
         0.00167596], dtype=float32)]}

In [16]:
# Neural models (Steps 4, 5a, 5b): saved sigmoid outputs, one file per seed, rows in load_pairs test order
SAVED = {
    "esim":  "../esim/checkpoints/esim_rand_mc2",
    "bi":    "../transformers/checkpoints",
    "cross": "../transformers/checkpoints_cross",
}
for model_id, folder in SAVED.items():
    probs[model_id] = [np.load(f"{folder}/probs_test_seed_{s}.npy").ravel() for s in SEEDS]

{m: [len(p) for p in ps] for m, ps in probs.items()}

{'xgb': [43844],
 'esim': [43844, 43844, 43844],
 'bi': [43844, 43844, 43844],
 'cross': [43844, 43844, 43844]}

In [17]:
# Check: the saved probabilities line up with these test rows
# (1) right length; (2) thresholding the probs reproduces the saved preds for every seed.
# Then compare the non-identical % found below with results.md before trusting the join.
for model_id, folder in SAVED.items():
    for s, p in zip(SEEDS, probs[model_id]):
        assert len(p) == len(test), (model_id, s, len(p), len(test))
        saved_preds = np.load(f"{folder}/preds_test_seed_{s}.npy").ravel()
        assert ((p >= 0.5).astype(int) == saved_preds).all(), (model_id, s)

non_identical = (test.neg_type == "case0") & (test.address_a != test.address_b)
pd.DataFrame({
    m: [round(100 * (p[non_identical] >= 0.5).mean(), 1) for p in ps] + [None] * (len(SEEDS) - len(ps))
    for m, ps in probs.items()
}, index = [f"seed {s}" for s in SEEDS])

,xgb,esim,bi,cross
seed 0,69.9,74.9,82.0,70.7
seed 1,NaN,78.9,83.0,73.0
seed 2,NaN,78.2,83.7,70.2


## 3. Game pool (TODO, David)
Filter `test` to the pairs the game shows: non-identical positives, `case2`, `case2s` (+ any easy tier). Optional: cap rows per unordered street pair.

In [18]:
# TODO: pool = test[...]
game_test_pool = test[test["neg_type"].isin(["case2", "case2s"]) | ((test["neg_type"] == "case0") & (test["address_a"] != test["address_b"]))]
game_test_pool = game_test_pool.groupby(["street_a", "street_b"]).head(2).reset_index().rename(columns = {"index": "test_row"})
game_test_pool

,test_row,id_a,id_b,address_a,address_b,zip_a,zip_b,split,neg_type,number_a,street_a,number_b,street_b,both_number_missing,number_missing,label
0,0,e21cb857-1fb6-4e3b-976e-10ef6b181f68,e21cb857-1fb6-4e3b-976e-10ef6b181f68,RED BARNS AT CORNER OF LITCHFIELD TURNPIKE CAR...,254 CARRINGTON RD,06524,06524,test,case0,NaN,RED BARNS AT CORNER OF LITCHFIELD TURNPIKE CAR...,254,CARRINGTON RD,False,True,1
1,5,781ef548-48b1-48da-9406-2a2e41371614,781ef548-48b1-48da-9406-2a2e41371614,41 VILLAGE LANE SUITE 201 UP AROUND THE BACK O...,41 VILLAGE LN,06524,06524,test,case0,41,VILLAGE LANE SUITE 201 UP AROUND THE BACK OF T...,41,VILLAGE LN,False,False,1
2,6,699ec1aa-2ccf-483e-a972-5845001b1e52,699ec1aa-2ccf-483e-a972-5845001b1e52,41 VILLAGE LN,41 VILLAGE LN 101-103,06524,06524,test,case0,41,VILLAGE LN,41,VILLAGE LN 101-103,False,False,1
3,26,40d68907-7f06-4e28-9ba8-cab1355ca0fb,40d68907-7f06-4e28-9ba8-cab1355ca0fb,160 BANK STREET,160 BANK ST,06483,06483,test,case0,160,BANK STREET,160,BANK ST,False,False,1
4,31,c339fbac-d43c-40d6-ba1e-a3eaac0ae4e4,c339fbac-d43c-40d6-ba1e-a3eaac0ae4e4,WESTWOOD PLAZA SHOPPING CENTER 2 KLARIDES VILL...,2 KLARIDES VILLAGE DR,06483,06483,test,case0,NaN,WESTWOOD PLAZA SHOPPING CENTER 2 KLARIDES VILL...,2,KLARIDES VILLAGE DR,False,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1399,43838,9b035885-e2f9-43ce-b0e1-7329c4ba95ea,8757e690-a77f-4d9d-9a4a-6a4e4d13ca5d,90 OLIVER ST,90 RIVER ST,06606,06606,test,case2s,90,OLIVER ST,90,RIVER ST,False,False,0
1400,43839,65fe314b-ee42-45b0-877c-204ac3a7964c,56281496-26fd-4c68-b306-efe03d3b6b1f,701 COTTAGE GROVE RD BLDG B,701 COTTAGE GROVE RD STE 103,06002,06002,test,case2s,701,COTTAGE GROVE RD BLDG B,701,COTTAGE GROVE RD STE 103,False,False,0
1401,43840,f3033a97-e995-463d-ad49-d0c3b03498c7,113969c0-4509-47a6-a0f3-5728bce510f2,1 SHAKER HILL RD,1 SHAKER RD,06082,06082,test,case2s,1,SHAKER HILL RD,1,SHAKER RD,False,False,0
1402,43841,93249a7c-b278-47eb-ba64-95e4e604198a,0e0e5655-e640-4e8d-a53e-4c047c2db948,1600 SOUTHEAST RD,1600 SOUTH RD,06032,06032,test,case2s,1600,SOUTHEAST RD,1600,SOUTH RD,False,False,0


In [19]:
game_test_pool["neg_type"].value_counts()

neg_type
case2s    727
case0     663
case2      14
Name: count, dtype: int64

## 4. One probability per model (TODO, David)
Decide: one seed, or the mean of the three seeds' probabilities.

In [20]:
ret = pd.DataFrame()
for model_id, folder in SAVED.items():
    logits = []
    for s, p in zip(SEEDS, probs[model_id]):
        assert len(p) == len(test), (model_id, s, len(p), len(test))
        saved_preds = np.load(f"{folder}/probs_test_seed_{s}.npy").ravel()
        logits.append(saved_preds)
    res = np.mean(logits, axis = 0)
    ret[f"{model_id}_logit"] = res
ret["xgb_logit"] = xgb.predict_proba(test_X)[:, 1]


In [21]:
merged_df = game_test_pool.merge(ret, left_on = "test_row", right_index = True, how = "left").drop (columns = "test_row")
merged_df

,id_a,id_b,address_a,address_b,zip_a,zip_b,split,neg_type,number_a,street_a,number_b,street_b,both_number_missing,number_missing,label,esim_logit,bi_logit,cross_logit,xgb_logit
0,e21cb857-1fb6-4e3b-976e-10ef6b181f68,e21cb857-1fb6-4e3b-976e-10ef6b181f68,RED BARNS AT CORNER OF LITCHFIELD TURNPIKE CAR...,254 CARRINGTON RD,06524,06524,test,case0,NaN,RED BARNS AT CORNER OF LITCHFIELD TURNPIKE CAR...,254,CARRINGTON RD,False,True,1,0.235497,0.078348,0.001252,0.873619
1,781ef548-48b1-48da-9406-2a2e41371614,781ef548-48b1-48da-9406-2a2e41371614,41 VILLAGE LANE SUITE 201 UP AROUND THE BACK O...,41 VILLAGE LN,06524,06524,test,case0,41,VILLAGE LANE SUITE 201 UP AROUND THE BACK OF T...,41,VILLAGE LN,False,False,1,0.980039,0.996372,0.921830,0.544573
2,699ec1aa-2ccf-483e-a972-5845001b1e52,699ec1aa-2ccf-483e-a972-5845001b1e52,41 VILLAGE LN,41 VILLAGE LN 101-103,06524,06524,test,case0,41,VILLAGE LN,41,VILLAGE LN 101-103,False,False,1,0.967517,0.999795,0.927866,0.931518
3,40d68907-7f06-4e28-9ba8-cab1355ca0fb,40d68907-7f06-4e28-9ba8-cab1355ca0fb,160 BANK STREET,160 BANK ST,06483,06483,test,case0,160,BANK STREET,160,BANK ST,False,False,1,0.999150,0.999997,0.996776,0.982625
4,c339fbac-d43c-40d6-ba1e-a3eaac0ae4e4,c339fbac-d43c-40d6-ba1e-a3eaac0ae4e4,WESTWOOD PLAZA SHOPPING CENTER 2 KLARIDES VILL...,2 KLARIDES VILLAGE DR,06483,06483,test,case0,NaN,WESTWOOD PLAZA SHOPPING CENTER 2 KLARIDES VILL...,2,KLARIDES VILLAGE DR,False,True,1,0.985561,0.999630,0.993681,0.839712
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1399,9b035885-e2f9-43ce-b0e1-7329c4ba95ea,8757e690-a77f-4d9d-9a4a-6a4e4d13ca5d,90 OLIVER ST,90 RIVER ST,06606,06606,test,case2s,90,OLIVER ST,90,RIVER ST,False,False,0,0.011684,0.000460,0.001160,0.067362
1400,65fe314b-ee42-45b0-877c-204ac3a7964c,56281496-26fd-4c68-b306-efe03d3b6b1f,701 COTTAGE GROVE RD BLDG B,701 COTTAGE GROVE RD STE 103,06002,06002,test,case2s,701,COTTAGE GROVE RD BLDG B,701,COTTAGE GROVE RD STE 103,False,False,0,0.194829,0.702254,0.035324,0.072626
1401,f3033a97-e995-463d-ad49-d0c3b03498c7,113969c0-4509-47a6-a0f3-5728bce510f2,1 SHAKER HILL RD,1 SHAKER RD,06082,06082,test,case2s,1,SHAKER HILL RD,1,SHAKER RD,False,False,0,0.338318,0.999666,0.720069,0.786047
1402,93249a7c-b278-47eb-ba64-95e4e604198a,0e0e5655-e640-4e8d-a53e-4c047c2db948,1600 SOUTHEAST RD,1600 SOUTH RD,06032,06032,test,case2s,1600,SOUTHEAST RD,1600,SOUTH RD,False,False,0,0.086037,0.527080,0.324530,0.650100


## 4b. LLM (zero-shot, OpenAI)
One call per pool pair. The model answers YES / NO **first**, then one sentence of reasoning.
- Probability: from the first token's log-probabilities, `p = p_yes / (p_yes + p_no)`.
- Reasoning: the rest of the text. It is written after the answer, so it is a justification, not the process that produced it.

Answers are cached to `LLM_CACHE` (keyed by the two address strings), so a crash or re-run never pays twice and the scores stay fixed.
Key: `OPENAI_API_KEY` in `ml-geocoding/.env` (git-ignored), never in this notebook.

In [22]:
# LLM setup: key, client, prompt, cache
import re
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../../.env")           # ml-geocoding/.env -> OPENAI_API_KEY
LLM_MODEL = "gpt-4o-mini"           # check OpenAI's current cheapest model that returns logprobs
LLM_CACHE = "llm_cache.parquet"
PROMPT = """Do these two address records refer to the same location?
A: {a}
B: {b}
Answer YES or NO first, then one short sentence explaining why."""

client = OpenAI()                   # reads OPENAI_API_KEY from the environment

CACHE_COLS = ["address_a", "address_b", "lp_yes", "lp_no", "reasoning"]
cache = pd.read_parquet(LLM_CACHE) if os.path.exists(LLM_CACHE) else pd.DataFrame(columns = CACHE_COLS)
assert list(cache.columns) == CACHE_COLS, "old cache format: delete llm_cache.parquet and re-run"
done = set(zip(cache.address_a, cache.address_b))
len(done)

0

In [23]:
# One pair -> (log-prob of YES, log-prob of NO, reasoning sentence)
# The log-probs come from the FIRST token only (None if not in its top 5); the reasoning is the text after YES/NO.
def score_pair(a, b):
    resp = client.chat.completions.create(
        model = LLM_MODEL,
        messages = [{"role": "user", "content": PROMPT.format(a = a, b = b)}],
        temperature = 0,
        max_tokens = 60,
        logprobs = True,
        top_logprobs = 5,
    )
    choice = resp.choices[0]
    lp = {"YES": None, "NO": None}
    for t in choice.logprobs.content[0].top_logprobs:   # "YES", "Yes", " yes" all count as YES; keep the most likely spelling
        key = t.token.strip().upper()
        if key in lp and lp[key] is None:
            lp[key] = t.logprob
    reasoning = re.sub(r"^\s*(YES|NO)\b[\s.,:;!-]*", "", choice.message.content or "", flags = re.I).strip()
    return lp["YES"], lp["NO"], reasoning

score_pair("41 VILLAGE LN", "41 VILLAGE LANE")       # smoke test: one paid call

(-3.814697265625e-06,
 -17.375003814697266,
 'both addresses refer to the same location as "LN" is a common abbreviation for "LANE."')

In [24]:
# Score every pool pair not yet in the cache; save every 50 calls
todo = [(a, b) for a, b in zip(merged_df.address_a, merged_df.address_b) if (a, b) not in done]
rows = []
for i, (a, b) in enumerate(todo):
    lp_yes, lp_no, reasoning = score_pair(a, b)
    rows.append({"address_a": a, "address_b": b, "lp_yes": lp_yes, "lp_no": lp_no, "reasoning": reasoning})
    if (i + 1) % 50 == 0 or i + 1 == len(todo):
        cache = pd.concat([cache, pd.DataFrame(rows)], ignore_index = True)
        cache.to_parquet(LLM_CACHE, index = False)
        done |= {(r["address_a"], r["address_b"]) for r in rows}
        rows = []
        print(f"{i + 1} / {len(todo)}")

/var/folders/f6/9gbrkrfj3p742lnj_c_3c44m0000gn/T/ipykernel_2625/3011985960.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache = pd.concat([cache, pd.DataFrame(rows)], ignore_index = True)


50 / 1404
100 / 1404
150 / 1404
200 / 1404
250 / 1404
300 / 1404
350 / 1404
400 / 1404
450 / 1404
500 / 1404
550 / 1404
600 / 1404
650 / 1404
700 / 1404
750 / 1404
800 / 1404
850 / 1404
900 / 1404
950 / 1404
1000 / 1404
1050 / 1404
1100 / 1404
1150 / 1404
1200 / 1404
1250 / 1404
1300 / 1404
1350 / 1404
1400 / 1404
1404 / 1404


In [25]:
# Log-probabilities -> one probability per pair; join probability + reasoning onto merged_df
# Missing token = probability 0. Both missing -> NaN (the section 5 check will catch it).
p_yes = np.exp(cache.lp_yes.astype(float)).fillna(0)
p_no = np.exp(cache.lp_no.astype(float)).fillna(0)
cache["llm_logit"] = (p_yes / (p_yes + p_no)).where(p_yes + p_no > 0)   # it's a probability; named like the other columns
cache["llm_reasoning"] = cache["reasoning"]

merged_df = merged_df.drop(columns = ["llm_logit", "llm_reasoning"], errors = "ignore").merge(
    cache[["address_a", "address_b", "llm_logit", "llm_reasoning"]].drop_duplicates(["address_a", "address_b"]),
    on = ["address_a", "address_b"], how = "left")
merged_df[["llm_logit", "llm_reasoning"]].head()

,llm_logit,llm_reasoning
0,0.029312,because the first address describes a location...
1,0.904651,both addresses refer to the same location as t...
2,0.999797,"both addresses refer to the same location, wit..."
3,1.000000,"both addresses refer to the same location, as ..."
4,0.999739,"both addresses refer to the same location, wit..."


## 5. Build + check + export (TODO, David)
Build `{"models": [...], "pairs": [...]}`, assert every pair's `probs` keys == the set of `models[].id` and every p is in [0, 1], then `json.dump` to `OUT_PATH`.

In [26]:
MODELS = [{"id": "xgb",   "name": "XGBoost",         "threshold": 0.5},
          {"id": "esim",  "name": "ESIM",            "threshold": 0.5},
          {"id": "bi",    "name": "Bi-encoder",      "threshold": 0.5},
          {"id": "cross", "name": "Cross-encoder",   "threshold": 0.5},
          {"id": "llm",   "name": "LLM (zero-shot)", "threshold": 0.5}]
ids = [m["id"] for m in MODELS]

# llm's probability sits in probs["llm"] like every other model; its reasoning is a pair-level field
pairs = [{"pair_id": f"{r.id_a}_{r.id_b}",
          "address_a": r.address_a, "address_b": r.address_b,
          "label": int(r.label), "neg_type": r.neg_type,
          "probs": {m: round(float(getattr(r, f"{m}_logit")), 4) for m in ids},
          "llm_reasoning": r.llm_reasoning}
         for r in merged_df.itertuples()]

assert all(set(p["probs"]) == set(ids) for p in pairs)                       # foreign key
assert all(0 <= v <= 1 for p in pairs for v in p["probs"].values())          # also catches NaN (e.g. an unscored LLM pair)
assert all(isinstance(p["llm_reasoning"], str) for p in pairs)               # every pair has reasoning

with open(OUT_PATH, "w") as f:
    json.dump({"models": MODELS, "pairs": pairs}, f)
len(pairs)

1404

In [29]:
merged_df.head()

,id_a,id_b,address_a,address_b,zip_a,zip_b,split,neg_type,number_a,street_a,...,street_b,both_number_missing,number_missing,label,esim_logit,bi_logit,cross_logit,xgb_logit,llm_logit,llm_reasoning
0,e21cb857-1fb6-4e3b-976e-10ef6b181f68,e21cb857-1fb6-4e3b-976e-10ef6b181f68,RED BARNS AT CORNER OF LITCHFIELD TURNPIKE CAR...,254 CARRINGTON RD,06524,06524,test,case0,NaN,RED BARNS AT CORNER OF LITCHFIELD TURNPIKE CAR...,...,CARRINGTON RD,False,True,1,0.235497,0.078348,0.001252,0.873619,0.029312,because the first address describes a location...
1,781ef548-48b1-48da-9406-2a2e41371614,781ef548-48b1-48da-9406-2a2e41371614,41 VILLAGE LANE SUITE 201 UP AROUND THE BACK O...,41 VILLAGE LN,06524,06524,test,case0,41,VILLAGE LANE SUITE 201 UP AROUND THE BACK OF T...,...,VILLAGE LN,False,False,1,0.980039,0.996372,0.921830,0.544573,0.904651,both addresses refer to the same location as t...
2,699ec1aa-2ccf-483e-a972-5845001b1e52,699ec1aa-2ccf-483e-a972-5845001b1e52,41 VILLAGE LN,41 VILLAGE LN 101-103,06524,06524,test,case0,41,VILLAGE LN,...,VILLAGE LN 101-103,False,False,1,0.967517,0.999795,0.927866,0.931518,0.999797,"both addresses refer to the same location, wit..."
3,40d68907-7f06-4e28-9ba8-cab1355ca0fb,40d68907-7f06-4e28-9ba8-cab1355ca0fb,160 BANK STREET,160 BANK ST,06483,06483,test,case0,160,BANK STREET,...,BANK ST,False,False,1,0.999150,0.999997,0.996776,0.982625,1.000000,"both addresses refer to the same location, as ..."
4,c339fbac-d43c-40d6-ba1e-a3eaac0ae4e4,c339fbac-d43c-40d6-ba1e-a3eaac0ae4e4,WESTWOOD PLAZA SHOPPING CENTER 2 KLARIDES VILL...,2 KLARIDES VILLAGE DR,06483,06483,test,case0,NaN,WESTWOOD PLAZA SHOPPING CENTER 2 KLARIDES VILL...,...,KLARIDES VILLAGE DR,False,True,1,0.985561,0.999630,0.993681,0.839712,0.999739,"both addresses refer to the same location, wit..."


In [31]:
model_logits = ["esim_logit", "bi_logit", "cross_logit", "xgb_logit", "llm_logit"]
n = len(merged_df)
for col in model_logits:
    print(f"{'='*20} {col} {'='*20}")
    positives = ((merged_df[col] >= 0.5).astype(int) ==  merged_df["label"]).sum()
    print(f"matched: {positives} || missed: {n - positives}")

==================== esim_logit ====================
matched: 1053 || missed: 351
==================== bi_logit ====================
matched: 976 || missed: 428
==================== cross_logit ====================
matched: 1118 || missed: 286
==================== xgb_logit ====================
matched: 1101 || missed: 303
==================== llm_logit ====================
matched: 1051 || missed: 353


## Diagnostics
One-off queries and their results (question → result → query).